# CTEs and subqueries: building queries in steps

A common table expression (WITH name AS (...)) names an intermediate result so a long query reads as a series of steps. Subqueries put a query inside another one, in FROM, in WHERE (IN, EXISTS) or as a single value. Both help you compute something first and use it next.

**How to run:** attach **Serverless** compute (top right), then run the cells from top to bottom. Each part asks you to predict the result first; the expected results are listed at the end.

From the Learning Hub lesson: [CTEs and subqueries: building queries in steps](https://www.datapreplab.com/hub#topic/sql-ctes-subqueries) · Independent study resource, not affiliated with or endorsed by Databricks.

**Goal:** solve a two-step question with CTEs, then compare IN, EXISTS and NOT IN.

**Part A: data.**

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (DATE'2026-01-01', 1, 100), (DATE'2026-01-01', 2, 50),
  (DATE'2026-01-02', 1, 40),
  (DATE'2026-01-03', 3, 200), (DATE'2026-01-03', NULL, 10)
AS t(order_date, customer_id, amount);

CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES (1, 'Asha'), (2, 'Ben'), (3, 'Chen'), (4, 'Dee') AS t(id, name);

**Part B: days above average.** Predict which days appear, then run:

In [ ]:
%sql
WITH daily AS (
  SELECT order_date, sum(amount) AS revenue
  FROM orders
  GROUP BY order_date
),
stats AS (
  SELECT avg(revenue) AS avg_revenue FROM daily
)
SELECT d.order_date, d.revenue
FROM daily d CROSS JOIN stats s
WHERE d.revenue > s.avg_revenue
ORDER BY d.order_date;

To check a step on its own, run just the `daily` query.

**Part C: the wrong grain.** Predict the difference:

In [ ]:
%sql
SELECT avg(amount) FROM orders;   -- 80, an average of orders, not of days

**Part D: NOT IN versus NOT EXISTS.** One order has a NULL customer_id. Predict each result:

In [ ]:
%sql
SELECT name FROM customers WHERE id NOT IN (SELECT customer_id FROM orders);       -- no rows
SELECT name FROM customers c
WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.id);               -- Dee

Expected evidence:

```text
Part B -> 2026-01-01 150 ; 2026-01-03 210
Part C -> 80.0
Part D -> NOT IN returns no rows ; NOT EXISTS returns Dee
```